In [1]:
import sys
from pathlib import Path

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

from lakehouse import Lakehouse

db = Lakehouse()
archivo_origen = db.archivo_origen
consultar = db.consultar
ejecutar = db.ejecutar

In [2]:
# 1. Comprobar estructura del CSV original
archivo = archivo_origen
assert archivo, "Falta DATABRICKS_RAW_CSV_PATH en .env"
estructura = consultar(f"""
SELECT COUNT(*) AS total_filas,
       COUNT_IF(size(split(value, ';')) = 22) AS filas_normales,
       COUNT_IF(size(split(value, ';')) = 23) AS ocupaciones_con_punto_y_coma,
       COUNT_IF(size(split(value, ';')) NOT IN (22, 23)) AS otras_estructuras
FROM read_files('{archivo}', format => 'text')
WHERE value NOT LIKE 'FecNac_Año;FecNac_Mes;%'
""")
print(estructura.to_string(index=False))
assert int(estructura.loc[0, 'total_filas']) == 5_095_731
assert int(estructura.loc[0, 'ocupaciones_con_punto_y_coma']) == 403
assert int(estructura.loc[0, 'otras_estructuras']) == 0


 total_filas  filas_normales  ocupaciones_con_punto_y_coma  otras_estructuras
     5095731         5095328                           403                  0


In [3]:
# 2. Reparar la ocupación partida por un ;
sql_staging = """CREATE OR REPLACE TABLE __CATALOGO__.staging.nacimientos_peru_reparado
USING DELTA AS
WITH lineas AS (
  SELECT value, split(value, ';') AS campos
  FROM read_files('__ARCHIVO_ORIGEN__', format => 'text')
  WHERE value NOT LIKE 'FecNac_Año;FecNac_Mes;%'
)
SELECT
  campos[0] AS fec_nac_anio,
  campos[1] AS fec_nac_mes,
  campos[2] AS peso_nacido,
  campos[3] AS talla_nacido,
  campos[4] AS dur_emb_parto,
  campos[5] AS condicion_parto,
  campos[6] AS sexo_nacido,
  campos[7] AS tipo_parto,
  campos[8] AS edad_madre,
  campos[9] AS estado_civil,
  campos[10] AS nivel_instruccion_madre,
  CASE WHEN size(campos) = 23 THEN concat(campos[11], ';', campos[12]) ELSE campos[11] END AS desc_ocupacion,
  CASE WHEN size(campos) = 23 THEN campos[13] ELSE campos[12] END AS num_embar_madre,
  CASE WHEN size(campos) = 23 THEN campos[14] ELSE campos[13] END AS hijos_vivo_madre,
  CASE WHEN size(campos) = 23 THEN campos[15] ELSE campos[14] END AS hijos_fallec_madre,
  CASE WHEN size(campos) = 23 THEN campos[16] ELSE campos[15] END AS nacmuer_abort_madre,
  CASE WHEN size(campos) = 23 THEN campos[17] ELSE campos[16] END AS pais_madre,
  CASE WHEN size(campos) = 23 THEN campos[18] ELSE campos[17] END AS id_ubigeo_inei,
  CASE WHEN size(campos) = 23 THEN campos[19] ELSE campos[18] END AS ipress,
  CASE WHEN size(campos) = 23 THEN campos[20] ELSE campos[19] END AS lugar_nacido,
  CASE WHEN size(campos) = 23 THEN campos[21] ELSE campos[20] END AS atiende_parto,
  CASE WHEN size(campos) = 23 THEN campos[22] ELSE campos[21] END AS financiador_parto,
  size(campos) AS num_campos_original,
  size(campos) = 23 AS ocupacion_reparada,
  '__NOMBRE_ARCHIVO__' AS archivo_origen
FROM lineas"""
ejecutar(sql_staging)
print("Tabla staging reparada")


Tabla staging reparada


In [4]:
# 3. Comprobar que no se perdieron filas
reparacion = consultar("""
SELECT COUNT(*) AS total_filas,
       COUNT_IF(ocupacion_reparada) AS filas_reparadas,
       MIN(num_campos_original) AS campos_min,
       MAX(num_campos_original) AS campos_max
FROM __CATALOGO__.staging.nacimientos_peru_reparado
""")
print(reparacion.to_string(index=False))
assert int(reparacion.loc[0, 'total_filas']) == 5_095_731
assert int(reparacion.loc[0, 'filas_reparadas']) == 403


 total_filas  filas_reparadas  campos_min  campos_max
     5095731              403          22          23


In [5]:
# 4. Revisar rangos y valores especiales
perfil = consultar("""
SELECT MIN(TRY_CAST(fec_nac_anio AS INT)) AS anio_min,
       MAX(TRY_CAST(fec_nac_anio AS INT)) AS anio_max,
       COUNT_IF(TRY_CAST(peso_nacido AS DOUBLE) IS NULL) AS peso_no_numerico,
       COUNT_IF(NOT TRY_CAST(peso_nacido AS DOUBLE) BETWEEN 500 AND 6000) AS peso_fuera_rango,
       COUNT_IF(TRY_CAST(edad_madre AS INT) IS NULL
                OR NOT TRY_CAST(edad_madre AS INT) BETWEEN 10 AND 60) AS edad_invalida,
       COUNT_IF(num_embar_madre = '>=5') AS embarazos_cinco_o_mas,
       COUNT_IF(hijos_fallec_madre = '-1') AS hijos_fallecidos_desconocido
FROM __CATALOGO__.staging.nacimientos_peru_reparado
""")
print(perfil.to_string(index=False))


 anio_min  anio_max  peso_no_numerico  peso_fuera_rango  edad_invalida  embarazos_cinco_o_mas  hijos_fallecidos_desconocido
     2015      2026                 0              3176             60                 489156                       4941908


In [6]:
# 5. Auditar formatos y categorías del diccionario
dominios = {
    'condicion_parto': ['EUTOCICO', 'CESAREA', 'INSTRUMENTADO', 'IGNORADO'],
    'sexo_nacido': ['MASCULINO', 'FEMENINO'],
    'tipo_parto': ['UNICO', 'DOBLE', 'TRIPLE', 'MAS DE TRES', 'IGNORADO'],
    'estado_civil': ['CASADO', 'CONVIVIENTE', 'DIVORCIADO', 'SEPARADO', 'SOLTERO', 'VIUDO', 'IGNORADO'],
    'nivel_instruccion_madre': ['IGNORADO', 'INICIAL/PRE-ESCOLAR', 'NINGUN NIVEL/ILETRADO',
                               'PRIMARIA COMPLETA', 'PRIMARIA INCOMPLETA', 'SECUNDARIA COMPLETA',
                               'SECUNDARIA INCOMPLETA', 'SUPERIOR NO UNIV. COMPLETA',
                               'SUPERIOR NO UNIV. INCOMPLETA', 'SUPERIOR UNIV. COMPLETA',
                               'SUPERIOR UNIV. INCOMPLETA'],
    'num_embar_madre': ['-1', '1', '2', '3', '4', '>=5'],
    'hijos_vivo_madre': ['-1', '1', '2', '3', '4', '>=5'],
    'hijos_fallec_madre': ['-1', '1', '2', '3', '4', '>=5'],
    'nacmuer_abort_madre': ['NINGUNO', 'IGNORADO', '1', '2', '3', '4', '5', '6',
                           '7', '8', '9', '10', '11 A MÁS'],
    'lugar_nacido': ['CENTRO DE TRABAJO', 'DOMICILIO', 'ESTABLECIMIENTO DE SALUD',
                     'IGNORADO', 'OTRO', 'VIA PUBLICA'],
    'atiende_parto': ['ENFERMERA(O)', 'FAMILIAR', 'IGNORADO', 'INTERNA(O)', 'MEDICO',
                     'MEDICO GINECO-OBSTETRA', 'NADIE(AUTOAYUDA)', 'OBSTETRA', 'OTRO',
                     'OTRO PROFESIONAL DE LA SALUD', 'PARTERA', 'PROMOTOR SALUD',
                     'TECNICO DE SALUD'],
    'financiador_parto': ['ESSALUD', 'IGNORADO', 'PARTICULAR', 'PRIVADOS', 'SANIDAD EP',
                          'SANIDAD FAP', 'SANIDAD NAVAL', 'SANIDAD PNP', 'SIS'],
}
columnas = [
    "COUNT(*) AS total_filas",
    "COUNT_IF(NOT fec_nac_anio RLIKE '^[0-9]{4}$') AS anio_formato_invalido",
    "COUNT_IF(NOT fec_nac_mes RLIKE '^[0-9]{2}$') AS mes_formato_invalido",
    "COUNT_IF(NOT peso_nacido RLIKE '^([0-9]{1,4}|-1)$') AS peso_formato_invalido",
    "COUNT_IF(NOT talla_nacido RLIKE '^([0-9]{1,3}([.][0-9])?|-1)$') AS talla_formato_invalido",
    "COUNT_IF(NOT dur_emb_parto RLIKE '^[0-9]{1,2}$') AS gestacion_formato_invalido",
    "COUNT_IF(NOT edad_madre RLIKE '^[0-9]{1,2}$') AS edad_formato_invalido",
    "COUNT_IF(NOT id_ubigeo_inei RLIKE '^([0-9]{6}|-1)$') AS ubigeo_formato_invalido",
    "COUNT_IF(NOT ipress RLIKE '^([0-9]{1,6}|-1)$') AS ipress_formato_invalido",
    "COUNT_IF(peso_nacido = '-1') AS peso_menos_uno",
    "COUNT_IF(talla_nacido = '-1') AS talla_menos_uno",
    "COUNT_IF(NOT TRY_CAST(fec_nac_mes AS INT) BETWEEN 1 AND 12) AS mes_fuera_rango",
    "COUNT_IF(NOT TRY_CAST(dur_emb_parto AS INT) BETWEEN 22 AND 44) AS gestacion_fuera_rango",
    "COUNT_IF(NOT TRY_CAST(NULLIF(TRIM(talla_nacido), '-1') AS DOUBLE) BETWEEN 30 AND 60) AS talla_fuera_rango",
    "COUNT_IF(id_ubigeo_inei RLIKE '^[0-9]{6}$' AND NOT substring(id_ubigeo_inei, 1, 2) BETWEEN '01' AND '25') AS ubigeo_departamento_invalido",
    "COUNT_IF(TRIM(hijos_fallec_madre) = '-1') AS hijos_fallecidos_menos_uno",
    "COUNT_IF(UPPER(TRIM(tipo_parto)) <> 'UNICO') AS partos_no_unicos",
    "COUNT_IF(UPPER(TRIM(sexo_nacido)) = 'IGNORADO') AS sexo_ignorado",
    "COUNT_IF(UPPER(TRIM(nacmuer_abort_madre)) = 'IGNORADO') AS abortos_ignorados",
    "COUNT_IF(UPPER(TRIM(nivel_instruccion_madre)) = 'INCIAL/PRE-ESCOLAR') AS instruccion_variante",
    "MAX(LENGTH(desc_ocupacion)) AS ocupacion_longitud_maxima",
    "COUNT_IF(TRIM(pais_madre) = '') AS pais_vacio",
    "COUNT_IF(num_campos_original NOT IN (22, 23)) AS filas_estructura_invalida"
]
for campo, valores in dominios.items():
    permitidos = ', '.join("'" + valor.replace("'", "''") + "'" for valor in valores)
    columnas.append(f'COUNT_IF(UPPER(TRIM({campo})) NOT IN ({permitidos})) AS {campo}_fuera_dominio')
sql_auditoria = ("CREATE OR REPLACE TABLE __CATALOGO__.metadata.auditoria_diccionario_nacimientos "
                 "USING DELTA AS SELECT " + ',\n       '.join(columnas) +
                 "\nFROM __CATALOGO__.staging.nacimientos_peru_reparado")
ejecutar(sql_auditoria)
auditoria = consultar('SELECT * FROM __CATALOGO__.metadata.auditoria_diccionario_nacimientos')
print(auditoria.to_string(index=False))
assert int(auditoria.loc[0, 'total_filas']) == 5_095_731
assert int(auditoria.loc[0, 'ubigeo_formato_invalido']) == 0
assert int(auditoria.loc[0, 'ipress_formato_invalido']) == 0

 total_filas  anio_formato_invalido  mes_formato_invalido  peso_formato_invalido  talla_formato_invalido  gestacion_formato_invalido  edad_formato_invalido  ubigeo_formato_invalido  ipress_formato_invalido  peso_menos_uno  talla_menos_uno  mes_fuera_rango  gestacion_fuera_rango  talla_fuera_rango  ubigeo_departamento_invalido  hijos_fallecidos_menos_uno  partos_no_unicos  sexo_ignorado  abortos_ignorados  instruccion_variante  ocupacion_longitud_maxima  pais_vacio  filas_estructura_invalida  condicion_parto_fuera_dominio  sexo_nacido_fuera_dominio  tipo_parto_fuera_dominio  estado_civil_fuera_dominio  nivel_instruccion_madre_fuera_dominio  num_embar_madre_fuera_dominio  hijos_vivo_madre_fuera_dominio  hijos_fallec_madre_fuera_dominio  nacmuer_abort_madre_fuera_dominio  lugar_nacido_fuera_dominio  atiende_parto_fuera_dominio  financiador_parto_fuera_dominio
     5095731                      0                     0                      0                      12                           

In [7]:
# 6. Contar filas idénticas en las 22 columnas originales
columnas_origen = ['fec_nac_anio', 'fec_nac_mes', 'peso_nacido', 'talla_nacido', 'dur_emb_parto',
                   'condicion_parto', 'sexo_nacido', 'tipo_parto', 'edad_madre', 'estado_civil',
                   'nivel_instruccion_madre', 'desc_ocupacion', 'num_embar_madre', 'hijos_vivo_madre',
                   'hijos_fallec_madre', 'nacmuer_abort_madre', 'pais_madre', 'id_ubigeo_inei',
                   'ipress', 'lugar_nacido', 'atiende_parto', 'financiador_parto']
duplicados = consultar(f"""
SELECT COUNT(*) AS combinaciones_repetidas,
       COALESCE(SUM(n - 1), 0) AS filas_repetidas
FROM (SELECT COUNT(*) AS n
      FROM __CATALOGO__.staging.nacimientos_peru_reparado
      GROUP BY {', '.join(columnas_origen)}
      HAVING COUNT(*) > 1)
""")
print(duplicados.to_string(index=False))
print('Se conservan: sin identificador no se distingue un duplicado de dos nacimientos con los mismos datos.')

 combinaciones_repetidas  filas_repetidas
                    1013             1023
Se conservan: sin identificador no se distingue un duplicado de dos nacimientos con los mismos datos.


In [8]:
# 7. Cargar la referencia de distritos: altitud, IDH 2019, pobreza y macrorregión
# Fuente: datos_referencia/ubigeo_distrito.csv (ubigeo-peru-aumentado, licencia MIT)
import pandas as pd

columnas_ref = {'inei': 'id_ubigeo_inei', 'provincia': 'provincia', 'distrito': 'distrito',
                'macroregion_minsa': 'macroregion', 'altitude': 'altitud_msnm', 'idh_2019': 'idh_2019',
                'pct_pobreza_total': 'pobreza_pct', 'pct_pobreza_extrema': 'pobreza_extrema_pct',
                'indice_vulnerabilidad_alimentaria': 'vulnerabilidad_alimentaria'}
ubigeos = (pd.read_csv(raiz / 'datos_referencia' / 'ubigeo_distrito.csv', dtype={'inei': str})
             [list(columnas_ref)].rename(columns=columnas_ref)
             .dropna(subset=['id_ubigeo_inei']))  # San Antonio (Moquegua) no tiene código INEI ni indicadores
assert ubigeos['id_ubigeo_inei'].is_unique, 'Un ubigeo repetido duplicaría nacimientos en el join'


def literal(valor):
    if pd.isna(valor):
        return 'NULL'
    if isinstance(valor, str):
        return "'" + valor.replace("'", "''") + "'"
    return repr(float(valor))


filas = ',\n'.join('(' + ', '.join(literal(v) for v in fila) + ')' for fila in ubigeos.itertuples(index=False))
ejecutar(f"""CREATE OR REPLACE TABLE __CATALOGO__.raw.ubigeo_distrito_referencia
USING DELTA AS
SELECT * FROM VALUES
{filas}
AS t({', '.join(ubigeos.columns)})""")
cargados = consultar('SELECT COUNT(*) AS n FROM __CATALOGO__.raw.ubigeo_distrito_referencia')
assert int(cargados.loc[0, 'n']) == len(ubigeos)
print('Distritos de referencia:', len(ubigeos))

Distritos de referencia: 1892


In [9]:
# 8. Crear tabla curada: originales en staging, desconocidos como nulos y marcas de validez
sql_curated = """CREATE OR REPLACE TABLE __CATALOGO__.curated.nacimientos_peru
USING DELTA AS
WITH tipados AS (
  SELECT
    TRY_CAST(fec_nac_anio AS INT) AS anio_nacimiento,
    TRY_CAST(fec_nac_mes AS INT) AS mes_nacimiento,
    TRY_CAST(NULLIF(TRIM(peso_nacido), '-1') AS INT) AS peso_nacido_g,
    TRY_CAST(NULLIF(TRIM(talla_nacido), '-1') AS DOUBLE) AS talla_nacido_cm,
    TRY_CAST(dur_emb_parto AS INT) AS duracion_embarazo_semanas,
    NULLIF(NULLIF(UPPER(TRIM(condicion_parto)), ''), 'IGNORADO') AS condicion_parto,
    NULLIF(NULLIF(UPPER(TRIM(sexo_nacido)), ''), 'IGNORADO') AS sexo_nacido,
    NULLIF(NULLIF(UPPER(TRIM(tipo_parto)), ''), 'IGNORADO') AS tipo_parto,
    TRY_CAST(edad_madre AS INT) AS edad_madre,
    NULLIF(NULLIF(UPPER(TRIM(estado_civil)), ''), 'IGNORADO') AS estado_civil,
    CASE WHEN UPPER(TRIM(nivel_instruccion_madre)) = 'INCIAL/PRE-ESCOLAR'
         THEN 'INICIAL/PRE-ESCOLAR'
         ELSE NULLIF(NULLIF(UPPER(TRIM(nivel_instruccion_madre)), ''), 'IGNORADO')
    END AS nivel_instruccion_madre,
    UPPER(NULLIF(TRIM(desc_ocupacion), '')) AS descripcion_ocupacion,
    NULLIF(NULLIF(TRIM(num_embar_madre), ''), '-1') AS num_embar_madre_cat,
    NULLIF(NULLIF(TRIM(hijos_vivo_madre), ''), '-1') AS hijos_vivos_madre_cat,
    -- El diccionario no tiene el valor 0: -1 (97 %) mezcla "ninguno" con "ignorado"
    CASE WHEN TRIM(hijos_fallec_madre) = '-1' THEN '0 O IGNORADO'
         ELSE NULLIF(TRIM(hijos_fallec_madre), '') END AS hijos_fallecidos_madre_cat,
    NULLIF(NULLIF(UPPER(TRIM(nacmuer_abort_madre)), ''), 'IGNORADO') AS antecedentes_abortos_cat,
    NULLIF(NULLIF(UPPER(TRIM(pais_madre)), ''), 'IGNORADO') AS pais_madre,
    NULLIF(NULLIF(TRIM(id_ubigeo_inei), ''), '-1') AS id_ubigeo_inei,
    CASE WHEN TRIM(id_ubigeo_inei) RLIKE '^[0-9]{6}$'
         THEN substring(TRIM(id_ubigeo_inei), 1, 2) END AS codigo_departamento,
    NULLIF(NULLIF(TRIM(ipress), ''), '-1') AS codigo_ipress,
    NULLIF(NULLIF(UPPER(TRIM(lugar_nacido)), ''), 'IGNORADO') AS lugar_nacido,
    CASE UPPER(TRIM(atiende_parto))
      WHEN 'ENFERMERA(O)' THEN 'ENFERMERA'
      WHEN 'INTERNA(O)' THEN 'INTERNO'
      WHEN 'NADIE(AUTOAYUDA)' THEN 'NADIE (AUTOAYUDA)'
      WHEN 'PROMOTOR SALUD' THEN 'PROMOTOR DE SALUD'
      ELSE NULLIF(NULLIF(UPPER(TRIM(atiende_parto)), ''), 'IGNORADO')
    END AS atiende_parto,
    NULLIF(NULLIF(UPPER(TRIM(financiador_parto)), ''), 'IGNORADO') AS financiador_parto,
    num_campos_original,
    ocupacion_reparada,
    archivo_origen
  FROM __CATALOGO__.staging.nacimientos_peru_reparado
), marcados AS (
  SELECT *,
         COALESCE(peso_nacido_g BETWEEN 500 AND 6000, FALSE) AS es_peso_valido,
         COALESCE(tipo_parto = 'UNICO', FALSE) AS es_parto_unico,
         COALESCE(duracion_embarazo_semanas BETWEEN 22 AND 44, FALSE) AS es_gestacion_valida,
         COALESCE(talla_nacido_cm BETWEEN 30 AND 60, FALSE) AS es_talla_valida,
         COALESCE(edad_madre BETWEEN 10 AND 60, FALSE) AS es_edad_madre_valida
  FROM tipados
), referencia AS (
  -- Cuartiles del peso por semana de gestación y sexo en partos únicos
  SELECT duracion_embarazo_semanas, sexo_nacido,
         percentile_approx(peso_nacido_g, 0.25) AS peso_q1,
         percentile_approx(peso_nacido_g, 0.75) AS peso_q3
  FROM marcados
  WHERE es_parto_unico AND es_peso_valido AND es_gestacion_valida
  GROUP BY duracion_embarazo_semanas, sexo_nacido
), completos AS (
  SELECT m.*,
         d.nombre AS departamento_residencia,
         u.provincia, u.distrito, u.macroregion, u.altitud_msnm, u.idh_2019,
         u.pobreza_pct, u.pobreza_extrema_pct, u.vulnerabilidad_alimentaria,
         CASE WHEN m.pais_madre IN ('PERU', 'VENEZUELA') THEN m.pais_madre
              WHEN m.pais_madre IS NOT NULL THEN 'OTRO' END AS pais_madre_grupo,
         CASE WHEN m.es_peso_valido THEN m.peso_nacido_g < 2500 END AS es_bajo_peso_nacer,
         -- Peso biológicamente inverosímil para la semana: fuera de 3 rangos intercuartílicos
         COALESCE(m.peso_nacido_g BETWEEN r.peso_q1 - 3 * (r.peso_q3 - r.peso_q1)
                                      AND r.peso_q3 + 3 * (r.peso_q3 - r.peso_q1), FALSE) AS es_peso_plausible_gestacion
  FROM marcados m
  LEFT JOIN referencia r
    ON m.duracion_embarazo_semanas = r.duracion_embarazo_semanas AND m.sexo_nacido = r.sexo_nacido
  LEFT JOIN (VALUES ('01', 'AMAZONAS'), ('02', 'ÁNCASH'), ('03', 'APURÍMAC'), ('04', 'AREQUIPA'),
                    ('05', 'AYACUCHO'), ('06', 'CAJAMARCA'), ('07', 'CALLAO'), ('08', 'CUSCO'),
                    ('09', 'HUANCAVELICA'), ('10', 'HUÁNUCO'), ('11', 'ICA'), ('12', 'JUNÍN'),
                    ('13', 'LA LIBERTAD'), ('14', 'LAMBAYEQUE'), ('15', 'LIMA'), ('16', 'LORETO'),
                    ('17', 'MADRE DE DIOS'), ('18', 'MOQUEGUA'), ('19', 'PASCO'), ('20', 'PIURA'),
                    ('21', 'PUNO'), ('22', 'SAN MARTÍN'), ('23', 'TACNA'), ('24', 'TUMBES'),
                    ('25', 'UCAYALI')) AS d(codigo, nombre)
    ON m.codigo_departamento = d.codigo
  LEFT JOIN __CATALOGO__.raw.ubigeo_distrito_referencia u
    ON m.id_ubigeo_inei = u.id_ubigeo_inei
)
SELECT *,
       -- Población de análisis: partos únicos con peso y gestación válidos y coherentes
       es_parto_unico AND es_peso_valido AND es_gestacion_valida
         AND es_peso_plausible_gestacion AS es_registro_analisis,
       CURRENT_TIMESTAMP() AS fecha_curacion
FROM completos"""
ejecutar(sql_curated)
print('Tabla curated creada')

Tabla curated creada


In [10]:
# 9. Validar la tabla curada con la auditoría del diccionario
calidad = consultar("""
SELECT COUNT(*) AS total_filas,
       COUNT_IF(ocupacion_reparada) AS ocupaciones_reparadas,
       COUNT_IF(es_peso_valido) AS pesos_validos,
       COUNT_IF(NOT es_peso_valido) AS pesos_invalidos,
       COUNT_IF(es_bajo_peso_nacer) AS bajo_peso,
       COUNT_IF(peso_nacido_g IS NULL) AS peso_sin_dato,
       COUNT_IF(talla_nacido_cm IS NULL) AS talla_sin_dato,
       COUNT_IF(sexo_nacido IS NULL) AS sexo_sin_dato,
       COUNT_IF(antecedentes_abortos_cat IS NULL) AS abortos_sin_dato,
       COUNT_IF(nivel_instruccion_madre = 'INICIAL/PRE-ESCOLAR') AS instruccion_normalizada,
       COUNT_IF(NOT es_edad_madre_valida OR edad_madre IS NULL) AS edades_invalidas,
       COUNT_IF(hijos_fallecidos_madre_cat = '0 O IGNORADO') AS hijos_fallecidos_cero_o_ignorado,
       COUNT_IF(es_parto_unico) AS partos_unicos,
       COUNT_IF(NOT es_gestacion_valida) AS gestacion_invalida,
       COUNT_IF(es_parto_unico AND es_peso_valido AND es_gestacion_valida
                AND NOT es_peso_plausible_gestacion) AS peso_inverosimil_gestacion,
       COUNT_IF(codigo_departamento IS NOT NULL AND departamento_residencia IS NULL) AS departamento_sin_nombre,
       COUNT_IF(id_ubigeo_inei IS NOT NULL AND altitud_msnm IS NULL) AS ubigeo_sin_referencia,
       COUNT_IF(es_registro_analisis) AS registros_analisis
FROM __CATALOGO__.curated.nacimientos_peru
""")
print(calidad.T.to_string(header=False))
assert int(calidad.loc[0, 'total_filas']) == int(auditoria.loc[0, 'total_filas'])
assert int(calidad.loc[0, 'ocupaciones_reparadas']) == 403
assert int(calidad.loc[0, 'pesos_validos']) + int(calidad.loc[0, 'pesos_invalidos']) == int(calidad.loc[0, 'total_filas'])
assert int(calidad.loc[0, 'partos_unicos']) + int(auditoria.loc[0, 'partos_no_unicos']) == int(calidad.loc[0, 'total_filas'])
for limpio, original in [('peso_sin_dato', 'peso_menos_uno'),
                         ('talla_sin_dato', 'talla_menos_uno'),
                         ('sexo_sin_dato', 'sexo_ignorado'),
                         ('abortos_sin_dato', 'abortos_ignorados'),
                         ('instruccion_normalizada', 'instruccion_variante'),
                         ('hijos_fallecidos_cero_o_ignorado', 'hijos_fallecidos_menos_uno'),
                         ('gestacion_invalida', 'gestacion_fuera_rango'),
                         ('departamento_sin_nombre', 'ubigeo_departamento_invalido')]:
    assert int(calidad.loc[0, limpio]) == int(auditoria.loc[0, original]), limpio
# Ubigeos que no están en la referencia (p. ej. distritos creados después): deben ser muy pocos
assert int(calidad.loc[0, 'ubigeo_sin_referencia']) < 0.001 * int(calidad.loc[0, 'total_filas'])

total_filas                       5095731
ocupaciones_reparadas                 403
pesos_validos                     5092555
pesos_invalidos                      3176
bajo_peso                          325716
peso_sin_dato                        1343
talla_sin_dato                       4826
sexo_sin_dato                         191
abortos_sin_dato                   432077
instruccion_normalizada               901
edades_invalidas                       60
hijos_fallecidos_cero_o_ignorado  4941908
partos_unicos                     4998923
gestacion_invalida                    289
peso_inverosimil_gestacion           3350
departamento_sin_nombre                 0
ubigeo_sin_referencia                3963
registros_analisis                4992312


In [11]:
# 10. Registrar control de calidad y origen
ejecutar("""
CREATE OR REPLACE TABLE __CATALOGO__.metadata.control_calidad_nacimientos
USING DELTA AS
SELECT CURRENT_TIMESTAMP() AS fecha_control,
       '__ARCHIVO_ORIGEN__' AS archivo_fuente,
       COUNT(*) AS total_filas,
       COUNT_IF(ocupacion_reparada) AS ocupaciones_reparadas,
       COUNT_IF(NOT es_peso_valido) AS pesos_invalidos,
       COUNT_IF(peso_nacido_g IS NULL) AS pesos_sin_dato,
       COUNT_IF(talla_nacido_cm IS NULL) AS tallas_sin_dato,
       COUNT_IF(sexo_nacido IS NULL) AS sexo_sin_dato,
       COUNT_IF(antecedentes_abortos_cat IS NULL) AS abortos_sin_dato,
       COUNT_IF(NOT es_gestacion_valida) AS gestacion_invalida,
       COUNT_IF(NOT es_parto_unico) AS partos_no_unicos,
       COUNT_IF(es_parto_unico AND es_peso_valido AND es_gestacion_valida
                AND NOT es_peso_plausible_gestacion) AS peso_inverosimil_gestacion,
       COUNT_IF(es_registro_analisis) AS registros_analisis,
       COUNT_IF(es_bajo_peso_nacer) AS bajo_peso
FROM __CATALOGO__.curated.nacimientos_peru
""")
print(consultar('SELECT * FROM __CATALOGO__.metadata.control_calidad_nacimientos').T.to_string(header=False))

fecha_control                                                                    2026-09-24 02:39:13.335579+00:00
archivo_fuente              /Volumes/workspace/raw/archivos_origen/DATOS_ABIERTOS_CNV_20260908_151033_5095731.csv
total_filas                                                                                               5095731
ocupaciones_reparadas                                                                                         403
pesos_invalidos                                                                                              3176
pesos_sin_dato                                                                                               1343
tallas_sin_dato                                                                                              4826
sexo_sin_dato                                                                                                 191
abortos_sin_dato                                                                        